Importing packages

In [2]:
import os
from dotenv import load_dotenv
import requests
import json
import pandas as pd
from datetime import datetime
import calendar
import time

Importing credentials

In [3]:
load_dotenv()

tmdb_token = os.getenv("tmdb_bearer_token")
omdb_api_key = os.getenv("omdb_api_key")

In [ ]:
discover_tmdb_url= 'https://api.themoviedb.org/3/discover/movie'

params = {'with_original_language':'en',
          'primary_release_date.gte':'2025-01-01',
          'primary_release_date.lte':'2025-12-31',
          'include_adult':'false',
          'include_video':'false',
          'sort_by':'primary_release_date.asc',
          'page':'1'
          }

headers = {'accept':'application/json',
           'Authorization':f'Bearer {tmdb_token}'
           }

response = requests.get(url=discover_tmdb_url, params=params, headers=headers)

page1_2025 = response.json()

print(json.dumps(page1_2025, indent=4))

In [ ]:
omdb_url = f'http://www.omdbapi.com/?apikey={omdb_api_key}&i=tt35538033&v=1'

response = requests.get(url = omdb_url)

omdb_test_data = response.json()

print(json.dumps(omdb_test_data, indent=4))

Both of my credentials stored in .env are working

Building a loop for collecting multi year ids

In [14]:
# Full year data works. Now testing all years

tmdb_ids = []
page_count = 1
current_year = datetime.now().year

for year in range(2000, current_year): #handles the loop to get movies from 2000 to whatever the latest complete year is
  for month in range(1,13): #handles each month in each year

    # If I partition by month I need a dynamic range for the first and last day
    first_day = 1 #month always starts with 1
    last_day = calendar.monthrange(year,month)[1] # the second value is the total days in the month
    

    """The responce in insomnia shows that the request returns a json and the movie data is in a list 
    called results. The first page is the default and it gives the total pages and total results for the particular 
    request. """
    discover_tmdb_url= 'https://api.themoviedb.org/3/discover/movie'

    params = {'with_original_language':'en',
              'primary_release_date.gte':f'{year}-{month:02d}-{first_day:02d}',
              'primary_release_date.lte':f'{year}-{month:02d}-{last_day}',
              'include_adult':'false',
              'include_video':'false',
              'sort_by':'primary_release_date.asc',
              'page': '1' # it defaults to one but its nice to have a visual
              }

    headers = {'accept':'application/json',
              'Authorization':f'Bearer {tmdb_token}'}

    initial_tmdb_response = requests.get(url=discover_tmdb_url, params=params, headers=headers, timeout=30)

    response.raise_for_status()
    page_data = initial_tmdb_response.json()



    for movie in page_data['results']:
      # collects the movie ids for the first page of the month
      movie_id = movie['id']
      tmdb_ids.append(movie_id) 

    print(f'{year}, Month: {month}, page: {page_count} complete')

    monthly_page_count = page_data['total_pages'] 
    

    for page in range(2, monthly_page_count + 1):

      #this makes the total pages result into a variable and gets pages 2 to the last 
    #page in the month request

      params = {'with_original_language':'en',
                #I am still in the same year and month so these variables dont need to change yet
              'primary_release_date.gte':f'{year}-{month:02d}-{first_day:02d}',
              'primary_release_date.lte':f'{year}-{month:02d}-{last_day}',
              'include_adult':'false',
              'include_video':'false',
              'sort_by':'primary_release_date.asc',
              'page': page # will start at 2 and keep going up until page count is hit range
              }

      second_response = requests.get(url=discover_tmdb_url, params=params, headers=headers, timeout=30)
      #second response is kinda a bad name for this variable, its misrepresentative
      #next response would be 

      response.raise_for_status()
      secondary_pages_data = second_response.json()

      for movie in secondary_pages_data['results']:
        movie_id = movie['id']
        tmdb_ids.append(movie_id)

      print(f'{year}, Month: {month}, page: {page} complete')

2000, Month: 1, page: 1 complete
2000, Month: 1, page: 2 complete
2000, Month: 1, page: 3 complete
2000, Month: 1, page: 4 complete
2000, Month: 1, page: 5 complete
2000, Month: 1, page: 6 complete
2000, Month: 1, page: 7 complete
2000, Month: 1, page: 8 complete
2000, Month: 1, page: 9 complete
2000, Month: 1, page: 10 complete
2000, Month: 1, page: 11 complete
2000, Month: 1, page: 12 complete
2000, Month: 1, page: 13 complete
2000, Month: 1, page: 14 complete
2000, Month: 1, page: 15 complete
2000, Month: 1, page: 16 complete
2000, Month: 1, page: 17 complete
2000, Month: 1, page: 18 complete
2000, Month: 1, page: 19 complete
2000, Month: 1, page: 20 complete
2000, Month: 1, page: 21 complete
2000, Month: 1, page: 22 complete
2000, Month: 1, page: 23 complete
2000, Month: 1, page: 24 complete
2000, Month: 1, page: 25 complete
2000, Month: 1, page: 26 complete
2000, Month: 1, page: 27 complete
2000, Month: 1, page: 28 complete
2000, Month: 1, page: 29 complete
2000, Month: 1, page: 3

In [15]:
print(len(tmdb_ids))

325123


In [16]:
print(len(set(tmdb_ids)))

325123
